# Пилот синтетического эксперимента

Цели проекта — подготовить сбор данных к A/B-тесту, сравнить четыре модели для предварительного выбора и проверить платформу. Здесь проектируем выборку и смотрим по одному примеру каждого сценария.

## Выборка

1 000 пользователей, 90 дней. В первые 60 дней выходят восемь публикаций. Следующие 30 дней продолжаем наблюдать продажи. Два курса — «Аналитика старт» и «Аналитика про». Исторические суммы по позициям этих курсов служат основой чеков. Они могут уже включать скидки или доли пакета, поэтому исторический прайс из них не восстанавливаем.

Одинаковые пользователи, расписание, затраты и заранее сгенерированные случайные числа во всех сценариях. Касания к недоступному продукту в сценарии запуска исключаются. Каждый пост указывает один курс. Категории — обычная публикация, скидка, анонс события и запуск. В пилоте все восемь размещений условно платные, по 6 000 ₽. Тип публикации хранится отдельно от способа размещения.

## Четыре сценария

| Сценарий | Что меняется относительно ровного спроса |
|---|---|
| Ровный спрос | Оба курса доступны. Базовая вероятность покупки каждого курса — 0,1% на пользователя в день. После рекламного касания к этому курсу прибавляем 0,3 п.п. в течение семи дней, включая день касания. |
| Внешнее событие | Дни 31–37: вероятность фоновой покупки «Аналитика про» умножается на 3. Публикации и рекламная прибавка те же. |
| Запуск курса | «Аналитика про» доступен с дня 31. Дни 31–37: фон для этого курса умножается на 3. Пост дня 31 получает категорию «запуск». Дата запуска условная. |
| Нулевой эффект | Ровный спрос и то же расписание касаний. Рекламная прибавка и снижение цены отключены. |

## Предположения генератора

- Вероятность касания каждого доступного размещения 16%. После рекламного касания вероятность просмотра 65%, после просмотра вероятность заявки 25%. У 30% пользователей в среднем есть один заход без метки.
- Заявки и оплаты генерируются отдельно. Таблица ниже считает заявки пользователей, которые совсем не купили за период. Для сверки оплат по конкретной заявке понадобится отдельная связка и время событий.
- Не больше двух заказов у пользователя, интервал более трёх дней. Фоновая вероятность повторной покупки уменьшается в четыре раза. Два курса в одном заказе возможны с вероятностью 10% при доступности обоих. Сумма заказа равна сумме позиций.
- Снижение цены на 15% действует для покупателя с касанием к скидочному посту о соответствующем курсе за последние семь дней. Скидка применяется к выбранной исторической сумме как условный сценарий. Рекламная прибавка к вероятности покупки одинакова для всех категорий.
- Шаг генерации — день. Внутри дня публикация и касание считаются доступными к моменту покупки. Эти таблицы предназначены для проверки выборки. Для отчёта бота понадобятся точные времена и связи со схемой БД.
- Семидневный срок воздействия в генераторе и 30-дневное окно атрибуции — разные параметры. Скрытая метка заказа описывает ветку генерации. В многокурсовом заказе она относится к покупке, инициировавшей заказ. Все вероятности, затраты и величины всплесков заданы для пилота.



In [1]:
from pathlib import Path
import sys

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'data/raw/base.csv').exists())
sys.path.insert(0, str(ROOT / 'src'))
from experiments.scenario_pilot import generate_pilots, summarize

pilots = dict(generate_pilots(ROOT / 'data/raw/base.csv', seed=42, n_users=1000))
summary = summarize(pilots.items())
summary


,scenario,users,visitors,touches,views,leads,leads_from_nonbuyers,buyers,orders,items,revenue_rub,ad_tagged_orders
0,flat,1000,835,1623,828,205,167,182,185,209,1542859.75,23
1,external_event,1000,835,1623,828,205,161,197,202,225,1672171.25,26
2,course_launch,1000,807,1458,715,176,142,162,166,184,1362644.75,17
3,zero_ad_effect,1000,835,1623,828,205,172,163,166,186,1384796.50,0


Один прогон помогает проверить объём, долю покупателей, наличие заявок без покупок и состав заказов. Для проверки календаря смотрим продажи по курсам и дням. Различия между четырьмя пилотами описывают условия симуляции. Следующий этап — проверка расчётов и отчёта бота, затем сравнение моделей на повторных прогонах.


In [2]:
import pandas as pd

daily_course_items = pd.concat([
    tables['items'].merge(tables['orders'][['order_id', 'day']], on='order_id', validate='many_to_one')
    .groupby(['day', 'course']).size().rename('items').reset_index().assign(scenario=name)
    for name, tables in pilots.items()
], ignore_index=True)
daily_course_items


,day,course,items,scenario
0,1,Аналитика старт,2,flat
1,2,Аналитика старт,2,flat
2,3,Аналитика про,1,flat
3,4,Аналитика старт,3,flat
4,5,Аналитика про,1,flat
...,...,...,...,...
455,88,Аналитика старт,1,zero_ad_effect
456,89,Аналитика про,1,zero_ad_effect
457,89,Аналитика старт,1,zero_ad_effect
458,90,Аналитика про,1,zero_ad_effect


## История и единица покупки

Строки одного студента с одинаковым точным временем образуют один заказ. Разные курсы внутри заказа остаются отдельными позициями с исходными суммами. Наличие нескольких курсов само по себе не подтверждает акцию. История используется для проверки единицы покупки и калибровки сумм. Атрибуцию запускаем на синтетике, где есть касания.


In [3]:
import numpy as np
import pandas as pd

from core.attribution import MODELS, compute
from core.importer import read_sales

history_items = read_sales(ROOT / 'data/raw/base.csv')
history_orders = history_items.groupby(['student_id', 'ts'], as_index=False).agg(
    amount=('amount', 'sum'), items=('course', 'size'),
    courses=('course', 'nunique'),
)
assert np.isclose(history_items.amount.sum(), history_orders.amount.sum(), atol=.005, rtol=0)
pd.Series({
    'Строки истории': len(history_items),
    'Заказы': len(history_orders),
    'Заказы с несколькими курсами': int(history_orders.courses.gt(1).sum()),
    'Выручка, ₽': round(history_orders.amount.sum(), 2),
})

Строки истории                      795.00
Заказы                              628.00
Заказы с несколькими курсами        153.00
Выручка, ₽                      5904671.67


## Прогон четырёх моделей

Синтетический день переводим в условную дату. Касания ставим на 10:00, оплаты на 18:00. Так сохраняем порядок, заданный генератором, включая покупку в день касания. Несколько касаний в одно время движок упорядочивает по идентификатору. Внутри каждого сценария все модели получают одну выборку и окно 30 дней.


In [4]:
def at_hour(days, hour):
    return pd.Timestamp('2025-01-01') + pd.to_timedelta(days - 1, unit='D') + pd.Timedelta(hours=hour)

def checked_attribution(touches, orders, model):
    result = compute(touches, orders, model=model, window_days=30)
    totals = result.groupby('order_id').agg(
        weight=('weight', 'sum'), revenue=('attributed_revenue', 'sum'),
    ).reindex(orders.order_id)
    np.testing.assert_allclose(totals.weight, 1, rtol=0, atol=1e-10)
    np.testing.assert_allclose(totals.revenue, orders.amount, rtol=0, atol=.005)

    matched = result.dropna(subset=['touch_id']).merge(
        touches[['touch_id', 'timestamp']], on='touch_id',
    ).merge(orders[['order_id', 'timestamp', 'previous']], on='order_id', suffixes=('_touch', '_order'))
    age = matched.timestamp_order - matched.timestamp_touch
    assert age.between(pd.Timedelta(0), pd.Timedelta(days=30)).all()
    assert (matched.previous.isna() | matched.timestamp_touch.gt(matched.previous)).all()
    return result

checks = []
placements_results = []
for scenario, tables in pilots.items():
    touches = tables['touches'].rename(columns={'user_id': 'user_id_hash'}).assign(
        timestamp=at_hour(tables['touches'].day, 10),
    )
    orders = tables['orders'].rename(columns={'user_id': 'user_id_hash'}).assign(
        timestamp=at_hour(tables['orders'].day, 18), variable_costs=0,
    ).sort_values(['user_id_hash', 'timestamp'])
    orders['previous'] = orders.groupby('user_id_hash').timestamp.shift()
    assert not orders.duplicated(['user_id_hash', 'timestamp']).any()
    item_totals = tables['items'].groupby('order_id').amount.sum().reindex(orders.order_id)
    np.testing.assert_allclose(item_totals, orders.amount, rtol=0, atol=.005)

    # Independent lookup of orders with at least one eligible placement touch.
    candidates = orders.merge(touches.dropna(subset=['placement_id']), on='user_id_hash',
                              suffixes=('_order', '_touch'))
    eligible = candidates.timestamp_touch.between(
        candidates.timestamp_order - pd.Timedelta(days=30), candidates.timestamp_order,
    ) & (candidates.previous.isna() | candidates.timestamp_touch.gt(candidates.previous))
    expected_unassigned = set(orders.order_id) - set(candidates.loc[eligible, 'order_id'])

    for model in MODELS:
        result = checked_attribution(touches, orders, model)
        unassigned = result.placement_id.isna()
        assert set(result.loc[unassigned, 'order_id']) == expected_unassigned
        revenue = result.groupby('placement_id').attributed_revenue.sum()
        placements = tables['placements'][['placement_id', 'cost']].copy()
        placements['revenue'] = placements.placement_id.map(revenue).fillna(0)
        placements['romi_pct'] = 100 * (placements.revenue - placements.cost) / placements.cost
        placements_results.append(placements.assign(scenario=scenario, model=model))
        checks.append({
            'scenario': scenario, 'model': model, 'orders': len(orders),
            'repeat_orders': int(orders.previous.notna().sum()),
            'orders_without_eligible_touch': int(unassigned.sum()),
            'total_rub': round(result.attributed_revenue.sum(), 2),
            'assigned_rub': round(placements.revenue.sum(), 2),
        })

checks = pd.DataFrame(checks)
placements_results = pd.concat(placements_results, ignore_index=True)
checks

          scenario     model  orders  repeat_orders  orders_without_eligible_touch   total_rub  assigned_rub
0             flat      last     185              3                            106  1542859.75     673873.25
1             flat     first     185              3                            106  1542859.75     673873.25
2             flat    linear     185              3                            106  1542859.75     673873.25
3             flat  position     185              3                            106  1542859.75     673873.25
4   external_event      last     202              5                            113  1672171.25     763824.75
5   external_event     first     202              5                            113  1672171.25     763824.75
6   external_event    linear     202              5                            113  1672171.25     763824.75
7   external_event  position     202              5                            113  1672171.25     763824.75
8    course_launch 

## Результаты

В сценарии с нулевым эффектом выключены и рекламная прибавка к вероятности покупки, и скидки. Выручка всё равно распределяется между размещениями, если перед покупкой были подходящие касания. Сравнение показывает чувствительность отчёта к модели на заданной выборке.


In [5]:
import matplotlib.pyplot as plt
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams.update({'font.size': 11, 'svg.fonttype': 'none'})
chart_dir = ROOT / 'outputs/experiments/scenario_pilot'
chart_dir.mkdir(parents=True, exist_ok=True)

scenario_names = {
    'flat': 'Ровный спрос', 'external_event': 'Внешнее событие',
    'course_launch': 'Запуск курса', 'zero_ad_effect': 'Нулевой эффект рекламы',
}
for metric, label in [('revenue', 'Приписанная выручка, ₽'), ('romi_pct', 'ROMI, %')]:
    fig, axes = plt.subplots(2, 2, figsize=(12, 7), sharey=True, layout='constrained')
    for ax, (scenario, name) in zip(axes.flat, scenario_names.items()):
        rows = placements_results.loc[placements_results.scenario.eq(scenario)]
        rows.pivot(index='placement_id', columns='model', values=metric)[list(MODELS)].plot.bar(
            ax=ax, legend=False, rot=0,
        )
        if metric == 'romi_pct':
            ax.axhline(0, color='gray', linewidth=1)
        ax.set(title=name, xlabel='Размещение', ylabel=label)
    handles, labels = axes[0, 0].get_legend_handles_labels()
    fig.legend(handles, labels, loc='outside upper center', ncol=4)
    fig.suptitle('Синтетика · 1 000 пользователей · 90 дней · seed 42', y=1.12)
    fig.savefig(chart_dir / f'{metric}.svg', bbox_inches='tight')
    fig.savefig(chart_dir / f'{metric}.png', dpi=150, bbox_inches='tight')
    plt.show()